In [6]:
import subprocess, json

In [7]:
jobs = json.loads(
    subprocess.check_output(["kubectl", "get", "jobs", "-n", "cms-ml", "-o", "json"], text=True)
)

In [8]:
deletes = []
for it in jobs.get("items", []):
    name = it["metadata"]["name"]
    if "bbtautau-" not in name:
        continue
    status = it.get("status", {})
    succeeded = status.get("succeeded", 0) > 0
    completed = any(
        c.get("type") == "Complete" and c.get("status") == "True"
        for c in (status.get("conditions") or [])
    )
    failed = status.get("failed", 0) > 0
    if succeeded or completed or failed:
        deletes.append(name)

In [9]:
deletes

[]

In [10]:
for job in deletes:
    subprocess.run(["kubectl", "delete", "job", job], check=False)